# 05 — AI Resume Matching (RAG): Data Preparation

Continúa desde [`01_eda.ipynb`](./01_eda.ipynb). Este notebook:

1. Repite las correcciones del EDA (BOM, columnas `responsibilities` ambiguas).
2. **Nuevo hallazgo, no visto en el EDA:** casi todos los campos estructurados del lado del CV vienen serializados como *strings* de listas de Python (a veces anidadas, a veces con `None`) — hay que parsearlos, no tratarlos como texto plano.
3. Deduplica a los **344 CVs únicos** y las **28 vacantes únicas** (hallazgo del EDA), separando el dataset en tres tablas: candidatos, vacantes, y pares con `matched_score`.
4. Construye el texto combinado de cada candidato y cada vacante, listo para embeddings en `03_modeling.ipynb`.

In [1]:
import ast

import pandas as pd

pd.set_option('display.max_columns', None)
DATA_PATH = '../data/resume_data.csv'

## 1. Carga y correcciones del EDA

In [2]:
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df.columns = [c.replace('\ufeff', '') for c in df.columns]
df = df.rename(columns={
    'responsibilities': 'candidate_past_responsibilities',
    'responsibilities.1': 'job_responsibilities',
})
print(df.shape)

(9544, 35)


## 2. Hallazgo nuevo: campos serializados como listas de Python

Columnas como `skills`, `degree_names`, `professional_company_names`, `related_skils_in_job`, `languages`, etc. no son texto plano — son el resultado de imprimir una lista de Python como string: `"['Big Data', 'Hadoop', 'Hive']"`, a veces anidada (`"[['Big Data']]"`) o llena de `None` (`'[None, None]'`). Si se usan tal cual para el embedding, el modelo de texto vería literalmente corchetes, comillas y la palabra `None` como si fueran contenido del CV — ruido que degrada la calidad del embedding.

In [3]:
LIST_LIKE_COLS = [
    'skills', 'educational_institution_name', 'degree_names', 'passing_years',
    'educational_results', 'result_types', 'major_field_of_studies',
    'professional_company_names', 'company_urls', 'start_dates', 'end_dates',
    'related_skils_in_job', 'positions', 'locations',
    'extra_curricular_activity_types', 'extra_curricular_organization_names',
    'extra_curricular_organization_links', 'role_positions', 'languages',
    'proficiency_levels', 'certification_providers', 'certification_skills',
    'online_links', 'issue_dates', 'expiry_dates',
]

def clean_list_field(raw) -> str:
    """Parsea un string tipo \"['A', 'B']\" (posiblemente anidado o con None)
    a texto plano separado por comas. `ast.literal_eval` es seguro acá —
    el contenido son literales de Python (listas/None/strings), no código
    ejecutable arbitrario."""
    if raw is None or (isinstance(raw, float)):
        return ''
    try:
        parsed = ast.literal_eval(str(raw))
    except (ValueError, SyntaxError):
        return str(raw).strip()

    def flatten(x):
        if isinstance(x, (list, tuple)):
            for item in x:
                yield from flatten(item)
        elif x is not None:
            yield str(x)

    parts = [p.strip() for p in flatten(parsed) if p and p.strip().upper() not in ('N/A', 'NONE')]
    return ', '.join(parts)

for col in LIST_LIKE_COLS:
    df[col] = df[col].apply(clean_list_field)

df[['skills', 'degree_names', 'related_skils_in_job', 'certification_skills']].head(3)

,skills,degree_names,related_skils_in_job,certification_skills
0,"Big Data, Hadoop, Hive, Python, Mapreduce, Spa...",B.Tech,Big Data,
1,"Data Analysis, Data Analytics, Business Analys...","B.Sc (Maths), M.Sc (Science) (Statistics)","Data Analysis, Business Analysis, Machine Lear...",
2,"Software Development, Machine Learning, Deep L...",B.Tech,"Unified Payment Interface, Risk Prediction, Bi...",


## 3. Deduplicar a candidatos y vacantes únicos

Confirmado en el EDA: 344 CVs únicos, 28 vacantes únicas. El fingerprint de candidato usa varios campos (no solo `career_objective` + `skills`) para minimizar el riesgo de colisión — se verifica que da el mismo resultado.

In [4]:
candidate_fp = (
    df['career_objective'].fillna('') + '|' + df['skills'] + '|' +
    df['educational_institution_name'] + '|' + df['degree_names'] + '|' +
    df['professional_company_names']
)
job_fp = df['job_position_name']

print('Candidatos únicos:', candidate_fp.nunique())
print('Vacantes únicas:', job_fp.nunique())

df['candidate_id'] = candidate_fp.factorize()[0]
df['job_id'] = job_fp.factorize()[0]

Candidatos únicos: 344
Vacantes únicas: 28


In [5]:
CANDIDATE_COLS = [
    'candidate_id', 'address', 'career_objective', 'skills',
    'educational_institution_name', 'degree_names', 'major_field_of_studies',
    'educational_results', 'professional_company_names', 'positions',
    'related_skils_in_job', 'candidate_past_responsibilities', 'languages',
    'proficiency_levels', 'certification_providers', 'certification_skills',
]
JOB_COLS = [
    'job_id', 'job_position_name', 'educationaL_requirements',
    'experiencere_requirement', 'age_requirement', 'job_responsibilities',
    'skills_required',
]

candidates = df[CANDIDATE_COLS].drop_duplicates(subset='candidate_id').reset_index(drop=True)
jobs = df[JOB_COLS].drop_duplicates(subset='job_id').reset_index(drop=True)
pairs = df[['candidate_id', 'job_id', 'matched_score']].copy()

print('candidates:', candidates.shape, '| jobs:', jobs.shape, '| pairs:', pairs.shape)

candidates: (344, 16) | jobs: (28, 7) | pairs: (9544, 3)


## 4. Perfil de texto combinado

Un solo campo de texto por candidato y por vacante, para vectorizar en `03_modeling.ipynb`.

In [6]:
def join_nonempty(*parts) -> str:
    clean = []
    for p in parts:
        if p is None or isinstance(p, float):  # covers NaN
            continue
        s = str(p).strip()
        if s:
            clean.append(s)
    return ' '.join(clean)

candidates['candidate_profile'] = candidates.apply(
    lambda r: join_nonempty(
        r['career_objective'], r['skills'], r['degree_names'], r['major_field_of_studies'],
        r['professional_company_names'], r['positions'], r['related_skils_in_job'],
        r['candidate_past_responsibilities'], r['languages'], r['certification_skills'],
    ), axis=1,
)

jobs['job_profile'] = jobs.apply(
    lambda r: join_nonempty(
        r['job_position_name'], r['educationaL_requirements'], r['experiencere_requirement'],
        r['skills_required'], r['job_responsibilities'],
    ), axis=1,
)

print('Ejemplo de perfil de candidato:')
print(candidates['candidate_profile'].iloc[0][:300])
print()
print('Ejemplo de perfil de vacante:')
print(jobs['job_profile'].iloc[0][:300])

Ejemplo de perfil de candidato:
Big data analytics working and database warehouse manager with robust experience in handling all kinds of data. I have also used multiple cloud infrastructure services and am well acquainted with them. Currently in search of role that offers more of development. Big Data, Hadoop, Hive, Python, Mapre

Ejemplo de perfil de vacante:
Senior Software Engineer B.Sc in Computer Science & Engineering from a reputed university. At least 1 year Technical Support
Troubleshooting
Collaboration
Documentation
System Monitoring
Software Deployment
Training & Mentorship
Industry Trends
Field Visits


## 5. Guardado de datasets procesados

In [7]:
candidates.to_csv('../data/processed_candidates.csv', index=False)
jobs.to_csv('../data/processed_jobs.csv', index=False)
pairs.to_csv('../data/processed_pairs.csv', index=False)

print('Guardado: processed_candidates.csv, processed_jobs.csv, processed_pairs.csv')

Guardado: processed_candidates.csv, processed_jobs.csv, processed_pairs.csv


## Conclusiones de la fase de Data Preparation

1. **Hallazgo nuevo resuelto:** los campos del CV venían serializados como listas de Python — se parsean con `ast.literal_eval` (seguro para literales, no código arbitrario), aplanando listas anidadas y descartando `None`/"N/A".
2. **Deduplicado a 344 candidatos y 28 vacantes**, separados en tres tablas: `processed_candidates.csv`, `processed_jobs.csv` y `processed_pairs.csv` (con `matched_score`, para validar el recomendador contra una referencia externa en `03_modeling.ipynb`).
3. **Un perfil de texto por candidato y por vacante**, listo para vectorizar — sin necesidad de una etiqueta de éxito para el recomendador content-based.

**Siguiente paso:** `03_modeling.ipynb` — pipeline de embeddings (`sentence-transformers`) + índice FAISS sobre los 344 candidatos, con validación cualitativa y cuantitativa (contra `matched_score`) del ranking.